# AbLang -- Antibody Language Model

![AbLang -- Antibody Language Model](https://proto-bio.github.io/proto-assets/images/tool/ablang/hero.png)

This notebook demonstrates how to use **AbLang** for antibody-specific embeddings, sequence scoring, and masked position restoration.

## What is AbLang?

AbLang is a family of antibody-specific language models trained on antibody sequence data from the Observed Antibody Space (OAS). It provides:

- **Embeddings** -- Antibody representations for similarity search, clustering, and downstream ML
- **Scoring** -- Pseudo-log-likelihood scoring to assess antibody sequence naturalness
- **Restore** -- Predict masked positions in antibody sequences (e.g., CDR loop reconstruction)

**Links:** [Paper (DOI: 10.1093/bioadv/vbae040)](https://doi.org/10.1093/bioadv/vbae040) | [GitHub](https://github.com/oxpig/AbLang2)

## Model Variants

AbLang wraps two generations of models under a single tool. The variant is selected automatically from the chains set on each `Antibody`:

| Model Variant | Model | Chains Provided | Embedding Dim | When to Use |
|---|---|---|---|---|
| `"ablang1-heavy"` | AbLang1 heavy | Heavy chain only | 768 | VH-only analysis |
| `"ablang1-light"` | AbLang1 light | Light chain only | 768 | VL-only analysis |
| `"ablang2-paired"` | AbLang2 | Heavy + light chains | 480 | Paired VH+VL analysis |

## Automatic Model Routing

There is no model setting: the tool inspects which chains each `Antibody` carries and picks the model:

- Only `heavy_chain` set → routes to **`ablang1-heavy`**
- Only `light_chain` set → routes to **`ablang1-light`**
- Both chains set → routes to **`ablang2-paired`**

## Imports

In [ ]:
from proto_tools.tools.masked_models.ablang import (
    run_ablang_embeddings,
    AbLangEmbeddingsInput,
    AbLangEmbeddingsConfig,
    run_ablang_score,
    AbLangScoringInput,
    AbLangScoringConfig,
    run_ablang_sample,
    AbLangSampleInput,
    AbLangSampleConfig,
    run_ablang_gradient,
    AbLangGradientInput,
    AbLangGradientConfig,
)
from proto_tools.entities.antibody import Antibody, AntibodyLogits
from proto_tools.utils import one_hot_protein_logits
from proto_tools.utils.notebook_docs import display_api_reference

## Input API Reference

The model variant is auto-resolved from the chains present on the `Antibody` / `AntibodyLogits` object: heavy-only -> `ablang1-heavy`, light-only -> `ablang1-light`, both -> `ablang2-paired`.

### `run_ablang_embeddings`

In [ ]:
display_api_reference("ablang", "input", "run_ablang_embeddings")

### `run_ablang_score`

In [ ]:
display_api_reference("ablang", "input", "run_ablang_score")

### `run_ablang_sample`

In [ ]:
display_api_reference("ablang", "input", "run_ablang_sample")

### `run_ablang_gradient`

In [ ]:
display_api_reference("ablang", "input", "run_ablang_gradient")

## Config API Reference

### `run_ablang_embeddings`

In [ ]:
display_api_reference("ablang", "config", "run_ablang_embeddings")

### `run_ablang_score`

In [ ]:
display_api_reference("ablang", "config", "run_ablang_score")

### `run_ablang_sample`

In [ ]:
display_api_reference("ablang", "config", "run_ablang_sample")

### `run_ablang_gradient`

In [ ]:
display_api_reference("ablang", "config", "run_ablang_gradient")

## Output API Reference

### `run_ablang_embeddings`

In [ ]:
display_api_reference("ablang", "output", "run_ablang_embeddings")

### `run_ablang_score`

In [ ]:
display_api_reference("ablang", "output", "run_ablang_score")

### `run_ablang_sample`

In [ ]:
display_api_reference("ablang", "output", "run_ablang_sample")

## 1. Extract Embeddings (auto-routing)

The tool detects single-chain vs paired antibodies automatically. Here we pass single heavy chains, so the tool routes to `ablang1-heavy` and returns 768-dim embeddings.

In [ ]:
# Trastuzumab (Herceptin) VH sequence
trastuzumab_vh = "EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLEWVARIYPTNGYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCSRWGGDGFYAMDYWGQGTLVTVSS"

# Pertuzumab VH sequence (another anti-HER2 antibody)
pertuzumab_vh = "EVQLVESGGGLVQPGGSLRLSCAASGFTFTDYTMDWVRQAPGKGLEWVADVNPNSGGSIYNQRFKGRFTLSVDRSKNTLYLQMNSLRAEDTAVYYCARNLGPSFYFDYWGQGTLVTVSS"

# Single-chain Antibody objects auto-route to ablang1-heavy
inputs = AbLangEmbeddingsInput(
    antibodies=[Antibody(heavy_chain=trastuzumab_vh), Antibody(heavy_chain=pertuzumab_vh)]
)
config = AbLangEmbeddingsConfig(batch_size=2)

embeddings_result = run_ablang_embeddings(inputs, config)

print(f"Model used: {embeddings_result.metadata['model_choice']}")
print(f"Processed {len(embeddings_result.results)} sequences")
print(f"Embedding dimension: {len(embeddings_result.results[0].mean_embedding)}")
print(f"Attention mask length (trastuzumab): {sum(embeddings_result.results[0].attention_mask)}")
print(f"Attention mask length (pertuzumab): {sum(embeddings_result.results[1].attention_mask)}")

## 2. Score Sequences

Compute pseudo-log-likelihood scores to assess how natural an antibody sequence looks to the model. Lower (more negative) scores indicate lower model confidence. This is useful for comparing natural antibodies against engineered or scrambled variants.

Again using auto-routing here — single chains route to `ablang1-heavy`.

In [ ]:
# Compare a natural antibody heavy chain vs a scrambled version
natural_vh = "EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLEWVARIYPTNGYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCSRWGGDGFYAMDYWGQGTLVTVSS"
# Scrambled version (same amino acids, random order — no non-standard residues)
scrambled_vh = "GVSTYGDRGYPQFSVWVDTAQYNRGISGLKRAAKEGSTYTIRRFLPVDSANDWTYGLPHVSGMTEQNLVCSAYGIKARLWPGNVTSELDFAGTKYREPCGQIHALNMSSWGTDAVYAEV"

# Single-chain Antibody objects auto-route to ablang1-heavy
inputs = AbLangScoringInput(
    antibodies=[Antibody(heavy_chain=natural_vh), Antibody(heavy_chain=scrambled_vh)]
)
config = AbLangScoringConfig(
    scoring_mode="pseudo_log_likelihood",
    batch_size=2,
)

score_result = run_ablang_score(inputs, config)

print("Average log-likelihood (higher = more natural):")
print(f"  Natural VH:   {score_result.scores[0].avg_log_likelihood:.3f}")
print(f"  Scrambled VH: {score_result.scores[1].avg_log_likelihood:.3f}")
print("Perplexity (lower = more natural):")
print(f"  Natural VH:   {score_result.scores[0].perplexity:.3f}")
print(f"  Scrambled VH: {score_result.scores[1].perplexity:.3f}")

## 3. Restore Masked Positions

Use AbLang to predict amino acids at masked positions (marked with `_`). This is especially useful for restoring CDR3 regions or filling in missing residues in antibody sequences.

In [ ]:
# Trastuzumab VH with CDR3 region masked (positions 99-110: SRWGGDGFYAMDY)
masked_vh = "EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLEWVARIYPTNGYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYC_____________WGQGTLVTVSS"

print(f"Original CDR3: SRWGGDGFYAMDY")
print(f"Masked input:  ...TAVYYC_____________WGQGTL...")

# Single-chain Antibody auto-routes to ablang1-heavy
inputs = AbLangSampleInput(antibodies=[Antibody(heavy_chain=masked_vh)])
sample_result = run_ablang_sample(inputs)

restored_seq = sample_result.sequences[0]
# Extract the restored CDR3 region
restored_cdr3 = restored_seq[97:110]
print(f"Restored CDR3: {restored_cdr3}")
print(f"Full restored: {restored_seq}")

## 4. Paired Heavy|Light Mode (auto-routing to ablang2-paired)

Set both `heavy_chain` and `light_chain` on a single `Antibody` and the tool auto-routes to `ablang2-paired`, which processes heavy and light chains together to capture inter-chain dependencies. The paired model produces 480-dim embeddings instead of 768-dim. Sample outputs for paired antibodies are returned as a single string `"heavy|light"` you can split on `|` to recover the two chains.

In [ ]:
# Trastuzumab paired heavy|light chains
trastuzumab_vh = "EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLEWVARIYPTNGYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCSRWGGDGFYAMDYWGQGTLVTVSS"
trastuzumab_vl = "DIQMTQSPSSLSASVGDRVTITCRASQDVNTAVAWYQQKPGKAPKLLIYSASFLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQHYTTPPTFGQGTKVEIK"

# Setting both chains on a single Antibody auto-routes to ablang2-paired
paired_ab = Antibody(heavy_chain=trastuzumab_vh, light_chain=trastuzumab_vl)

inputs = AbLangEmbeddingsInput(antibodies=[paired_ab])
paired_result = run_ablang_embeddings(inputs)
print(f"Model used: {paired_result.metadata['model_choice']}")
print(f"Paired embedding dimension: {len(paired_result.results[0].mean_embedding)}")

# Scoring — also auto-routes to ablang2-paired
inputs = AbLangScoringInput(antibodies=[paired_ab])
config = AbLangScoringConfig(scoring_mode="pseudo_log_likelihood")
paired_score = run_ablang_score(inputs, config)
print(f"Paired log-likelihood: {paired_score.scores[0].avg_log_likelihood:.3f}")

# Restore masked positions in paired mode (mask CDR3 of light chain)
masked_vl = "DIQMTQSPSSLSASVGDRVTITCRASQDVNTAVAWYQQKPGKAPKLLIYSASFLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYC_________FGQGTKVEIK"
inputs = AbLangSampleInput(antibodies=[Antibody(heavy_chain=trastuzumab_vh, light_chain=masked_vl)])
paired_restore = run_ablang_sample(inputs)
restored_heavy, restored_light = paired_restore.sequences[0].split("|")
print(f"Original VL CDR3: QQHYTTPPT")
print(f"Restored VL:      {restored_light}")

## 5. Light-Chain Routing

A light-chain-only `Antibody` (only `light_chain` set) auto-routes to `ablang1-light` — no extra setting is needed.

In [ ]:
# Light-only Antibody (no heavy chain) auto-routes to ablang1-light
trastuzumab_vl = "DIQMTQSPSSLSASVGDRVTITCRASQDVNTAVAWYQQKPGKAPKLLIYSASFLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQHYTTPPTFGQGTKVEIK"

inputs = AbLangEmbeddingsInput(antibodies=[Antibody(light_chain=trastuzumab_vl)])

light_result = run_ablang_embeddings(inputs)

print(f"Model used: {light_result.metadata['model_choice']}")
print(f"Embedding dimension: {len(light_result.results[0].mean_embedding)}")

## 6. Compute Gradients

`run_ablang_gradient` differentiates AbLang's masked pseudo-log-likelihood objective with respect to a relaxed `(L, 20)` distribution over amino acids (wrapped in an `AntibodyLogits` entity, which can carry a heavy chain, a light chain, or both). Use this as a differentiable, antibody-specific pLM loss inside MCMC or gradient descent over relaxed sequences. Set `compute_gradient=False` for forward-only PLL scoring without the backward pass.

In [ ]:
# Build relaxed logits for both chains. one_hot_protein_logits with sharpness=2.0
# yields a biased-but-not-saturated softmax target — a reasonable seed for MCMC.
heavy_logits = one_hot_protein_logits(trastuzumab_vh, sharpness=2.0)
light_logits = one_hot_protein_logits(trastuzumab_vl, sharpness=2.0)
antibody = AntibodyLogits(heavy_chain=heavy_logits, light_chain=light_logits)

gradient_input = AbLangGradientInput(inputs=antibody)
gradient_config = AbLangGradientConfig(
    hard=0.0,                # Pure softmax; hard=1.0 for a straight-through estimator
    compute_gradient=True,   # Set False for forward-only scoring
    device="cuda",           # Change to "cpu" if no GPU available
)

gradient_result = run_ablang_gradient(gradient_input, gradient_config)

# One result per antibody; this call passed one, so results holds one
candidate = gradient_result.results[0]
term = candidate.terms[0]

print(f"Mean NLL:           {term.term_value:.3f}")
print(f"Log-likelihood:     {candidate.metrics['log_likelihood']:.3f}")
print(f"Sequence length:    {candidate.metrics['sequence_length']}")
print(f"Model choice:       {candidate.metrics['model_choice']}")
assert term.gradient is not None
print(f"Gradient shape:     ({len(term.gradient)}, {len(term.gradient[0])})")
print(f"Gradient row 0 (first 5 cols): {[round(v, 4) for v in term.gradient[0][:5]]}")

## 7. Export Results

Save results to files for downstream analysis.

### Supported export formats

| Output type | Formats |
|---|---|
| Embeddings | `csv`, `json`, `npy`, `pt` |
| Scoring | `csv`, `json` |
| Sampling | `fasta`, `txt`, `json` |

In [ ]:
# Export embeddings to CSV
embeddings_result.export("ablang_embeddings", export_path="./example_output", file_format="csv")
print("Embeddings exported to ./example_output/ablang_embeddings.csv")

# Export scores to JSON
score_result.export("ablang_scores", export_path="./example_output", file_format="json")
print("Scores exported to ./example_output/ablang_scores.json")

# Export restored sequences to FASTA
sample_result.export("ablang_restored", export_path="./example_output", file_format="fasta")
print("Restored sequences exported to ./example_output/ablang_restored.fasta")